# 07 -- Organização e dataclasses

**GA033 · Módulo introdutório de programação científica**  
Autor: Diego Tavares Volpatto · Licença: [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/)

Uma malha de intervalos nos dá um exemplo pequeno para organizar dados científicos.
Começaremos com funções e arrays, importaremos uma função de um módulo e
reuniremos dados e operações em uma classe. Por fim, usaremos uma `dataclass`
com validação explícita.

Pré-requisitos: funções, arrays, índices e operações NumPy. Não é necessário
conhecer programação orientada a objetos. O exemplo apenas representa uma malha;
não monta nem resolve um problema de elementos finitos.

Ao final, você deverá explicar `self`, atributo de instância, método, anotação de
tipo e validação explícita. Execute de cima para baixo e confira com **Restart Kernel and Run All**.


In [1]:
from dataclasses import dataclass
from pathlib import Path
import sys

import numpy as np

## Primeiro, uma função para uma tarefa

Uma função tem entradas e devolve um resultado. Para uma malha uniforme, o número
de nós é o número de elementos mais um. Aqui usamos entradas válidas para entender
a construção; o módulo completo, adiante, também verificará entradas inválidas.


In [2]:
def gerar_coordenadas(numero_elementos, comprimento=1.0):
    return np.linspace(0.0, comprimento, numero_elementos + 1)


coordenadas = gerar_coordenadas(4)
print(coordenadas)

[0.   0.25 0.5  0.75 1.  ]


## Quais nós pertencem a cada elemento?

Cada linha da **conectividade** contém os índices do nó inicial e do nó final.
Por exemplo, `[1, 2]` usa `coordenadas[1]` e `coordenadas[2]`.
`np.column_stack` reúne duas colunas; `[:, 0]` seleciona a primeira coluna.


In [3]:
nos_iniciais = np.arange(4)
conectividade = np.column_stack((nos_iniciais, nos_iniciais + 1))
comprimentos = coordenadas[conectividade[:, 1]] - coordenadas[conectividade[:, 0]]
print("Conectividade:")
print(conectividade)
print("Comprimentos:", comprimentos)

Conectividade:
[[0 1]
 [1 2]
 [2 3]
 [3 4]]
Comprimentos: [0.25 0.25 0.25 0.25]


## Antes da classe: importar uma função de um módulo

Um **módulo** é um arquivo Python que podemos importar de outros programas.
O arquivo [`exemplos/calculos.py`](../exemplos/calculos.py) guarda a função
`calcular_comprimentos`. Ela faz a mesma operação que acabamos de escrever,
recebendo os dois arrays como argumentos. Abra o arquivo e identifique essa função.

A célula seguinte localiza a raiz do projeto quando iniciamos pela raiz ou por
`notebooks/`. `sys.path` indica onde esta sessão procura módulos; acrescentar a
raiz permite importar `exemplos`. Isso não instala um pacote nem altera seu Python global.


In [4]:
raiz = Path.cwd()
if raiz.name == "notebooks":
    raiz = raiz.parent
if not (raiz / "pixi.toml").is_file():
    raise RuntimeError("Inicie pela raiz do repositório ou pela pasta notebooks.")
if str(raiz) not in sys.path:
    sys.path.insert(0, str(raiz))

from exemplos.calculos import calcular_comprimentos

print("Módulo usado: exemplos/calculos.py")
print("Comprimentos:", calcular_comprimentos(coordenadas, conectividade))

Módulo usado: exemplos/calculos.py
Comprimentos: [0.25 0.25 0.25 0.25]


## Uma classe reúne dados relacionados e operações

Uma **instância** é um objeto criado a partir da classe. Seus **atributos** guardam
dados próprios. Um **método** é uma função associada a esse objeto.

Na classe didática abaixo, `__init__` recebe os dados ao criar o objeto.
`self` representa a instância: `self.coordenadas` é o vetor guardado nela.
O método `comprimentos` reutiliza a função que acabamos de importar. Esta
primeira versão ainda não valida entradas; vamos corrigir isso adiante.


In [5]:
class MalhaDidatica:
    def __init__(self, coordenadas, conectividade):
        self.coordenadas = np.array(coordenadas, dtype=float, copy=True)
        self.conectividade = np.array(conectividade, copy=True)

    def comprimentos(self):
        return calcular_comprimentos(self.coordenadas, self.conectividade)


malha_didatica = MalhaDidatica(coordenadas, conectividade)
print(malha_didatica.comprimentos())

[0.25 0.25 0.25 0.25]


Ao escrever `malha_didatica.comprimentos()`, Python fornece a instância como
`self`. Por isso não passamos `self` manualmente nessa chamada.
Objetos da mesma classe podem guardar dados diferentes.


In [6]:
outra_malha = MalhaDidatica(gerar_coordenadas(4, comprimento=2.0), conectividade)
print("Primeira instância:", malha_didatica.comprimentos())
print("Outra instância:   ", outra_malha.comprimentos())

Primeira instância: [0.25 0.25 0.25 0.25]
Outra instância:    [0.5 0.5 0.5 0.5]


## `dataclass` reduz a escrita repetitiva

`@dataclass` gera, entre outras coisas, um `__init__` e uma representação textual.
As linhas `coordenadas: np.ndarray` e `conectividade: np.ndarray` são **anotações
de tipo**: comunicam a intenção, mas não validam automaticamente os dados.

Usamos `eq=False` porque a comparação automática de campos não é apropriada
para arrays NumPy. Para comparar os dados, podemos usar `np.array_equal` ou
`np.allclose`, conforme o que queremos verificar.


In [7]:
@dataclass(eq=False)
class MalhaEmConstrucao:
    coordenadas: np.ndarray
    conectividade: np.ndarray

    def comprimentos(self):
        return calcular_comprimentos(self.coordenadas, self.conectividade)


exemplo_incompleto = MalhaEmConstrucao(np.array([0.0, 1.0]), np.array([[1, 0]]))
print(exemplo_incompleto)
print("Comprimento obtido:", exemplo_incompleto.comprimentos())

MalhaEmConstrucao(coordenadas=array([0., 1.]), conectividade=array([[1, 0]]))
Comprimento obtido: [-1.]


O comprimento negativo denuncia um elemento invertido: a anotação não impediu
sua construção. Uma estrutura útil precisa explicitar suas garantias.

O módulo [`exemplos/malha.py`](../exemplos/malha.py) contém a versão `Malha1D`,
que acrescenta validação e reutiliza `calcular_comprimentos`. Vamos importar
essa versão para os próximos exemplos.


In [8]:
from exemplos.malha import Malha1D, criar_malha_uniforme

print("Módulo usado: exemplos/malha.py")

Módulo usado: exemplos/malha.py


## Validação explícita em `__post_init__`

No módulo, `__post_init__` é chamado depois do `__init__` gerado por `dataclass`.
Ele verifica coordenadas reais, finitas e em um vetor com pelo menos dois nós;
conectividade inteira com pelo menos uma linha, duas colunas e índices válidos; e comprimento positivo
de cada elemento. Também copia os arrays recebidos.

Isso verifica estrutura e orientação local. Não é uma prova de que elementos
arbitrários cubram um domínio sem lacunas ou sobreposições.


In [9]:
malha = criar_malha_uniforme(numero_elementos=4, comprimento=1.0)
print(malha)
print("Comprimentos:", malha.comprimentos())

Malha1D(coordenadas=array([0.  , 0.25, 0.5 , 0.75, 1.  ]), conectividade=array([[0, 1],
       [1, 2],
       [2, 3],
       [3, 4]]))
Comprimentos: [0.25 0.25 0.25 0.25]


## Copiar entradas evita uma surpresa frequente

Se uma classe guarda a mesma referência de um array externo, uma edição nesse
array pode mudar o objeto sem que isso esteja claro no código. Nossa classe
copia as entradas. O experimento abaixo altera somente o array original.


In [10]:
entrada_coordenadas = np.array([0.0, 0.4, 1.0])
entrada_conectividade = np.array([[0, 1], [1, 2]])
malha_copiada = Malha1D(entrada_coordenadas, entrada_conectividade)
entrada_coordenadas[1] = 0.9
print("Array externo após a mudança:", entrada_coordenadas)
print("Coordenadas guardadas:       ", malha_copiada.coordenadas)

Array externo após a mudança: [0.  0.9 1. ]
Coordenadas guardadas:        [0.  0.4 1. ]


A cópia não torna o objeto imutável. Seus atributos continuam acessíveis e podem
ser alterados. A validação ocorre **na construção**; não prometemos revalidar toda
edição posterior. Neste exemplo, prefira construir uma nova malha ao mudar seus dados.

Podemos verificar um erro esperado sem interromper o restante do notebook.
A mensagem abaixo confirma que um índice inexistente foi recusado.


In [11]:
try:
    Malha1D(np.array([0.0, 1.0]), np.array([[0, 2]]))
except ValueError as erro_esperado:
    print("Erro esperado:", erro_esperado)

Erro esperado: A conectividade contém um índice fora do vetor de coordenadas.


## Pequenas verificações com significado

Na malha uniforme construída, os comprimentos devem ser iguais e sua soma deve
coincidir com o comprimento do intervalo. `np.allclose` e `np.isclose` comparam
números de ponto flutuante com tolerâncias; igualdade exata nem sempre é adequada.


In [12]:
malha_verificacao = criar_malha_uniforme(3, comprimento=2.0)
tamanhos = malha_verificacao.comprimentos()
assert np.allclose(tamanhos, 2.0 / 3.0)
assert np.isclose(tamanhos.sum(), 2.0)
print("Verificações passaram:", tamanhos)

Verificações passaram: [0.66666667 0.66666667 0.66666667]


## Exercício 1 -- Construir outra malha uniforme

Altere os parâmetros abaixo para construir **6 elementos em um intervalo de
comprimento 3**. Antes de executar, determine o número de nós e o comprimento
de cada elemento. O código inicial já funciona com um exemplo diferente.


In [13]:
numero_elementos = 4  # Troque para o valor solicitado.
comprimento = 1.0  # Troque para o valor solicitado.
malha_exercicio = criar_malha_uniforme(numero_elementos, comprimento)
print("Número de nós:", malha_exercicio.coordenadas.size)
print("Comprimentos:", malha_exercicio.comprimentos())

Número de nós: 5
Comprimentos: [0.25 0.25 0.25 0.25]


### Solução comentada do exercício 1

São 7 nós e 6 comprimentos iguais a $3/6=0{,}5$. O teste verifica o resultado
esperado, em vez de apenas repetir as instruções de construção.


In [14]:
malha_solucao = criar_malha_uniforme(6, comprimento=3.0)
assert malha_solucao.coordenadas.size == 7
np.testing.assert_allclose(malha_solucao.comprimentos(), 0.5)
print(malha_solucao.comprimentos())

[0.5 0.5 0.5 0.5 0.5 0.5]


## Exercício 2 -- Uma malha não uniforme

Use os nós `[0.0, 0.2, 0.7, 1.0]` e ligue nós consecutivos. Construa um objeto
`Malha1D`, calcule os três comprimentos e identifique o maior. A classe já sabe
fazer isso; não é preciso criar outra classe ou mudar seu método.


In [15]:
coordenadas_exercicio = np.array([0.0, 0.2, 0.7, 1.0])
conectividade_exercicio = np.array([[0, 1], [1, 2], [2, 3]])
# Complete: malha_nao_uniforme = Malha1D(...)
# Depois, use .comprimentos() e np.max(...).
print("Dados preparados; complete a construção da malha e o cálculo.")

Dados preparados; complete a construção da malha e o cálculo.


### Solução comentada do exercício 2

As diferenças são 0,2; 0,5; 0,3. O maior comprimento é 0,5 e a soma é 1.
A representação por coordenadas e conectividade funciona para elementos de
tamanhos diferentes.


In [16]:
malha_nao_uniforme = Malha1D(coordenadas_exercicio, conectividade_exercicio)
tamanhos_nao_uniformes = malha_nao_uniforme.comprimentos()
np.testing.assert_allclose(tamanhos_nao_uniformes, [0.2, 0.5, 0.3])
print("Comprimentos:", tamanhos_nao_uniformes)
print("Maior comprimento:", np.max(tamanhos_nao_uniformes))

Comprimentos: [0.2 0.5 0.3]
Maior comprimento: 0.49999999999999994


## O módulo também tem testes fora do notebook

Na raiz do projeto, execute `pixi run testar`. Os testes de
[`testes/test_malha.py`](../testes/test_malha.py) verificam uma malha conhecida,
a função importada, a cópia das entradas e a recusa de malhas vazias, formatos,
índices e comprimentos inválidos.
Eles não substituem todos os cuidados de modelagem; documentam garantias concretas da interface.

Não precisamos usar classes para toda tarefa. Uma função basta quando representa
bem a operação. Aqui a classe ajuda porque coordenadas, conectividade e cálculo
de comprimentos pertencem ao mesmo objeto.

Referências: [`dataclasses` no Python](https://docs.python.org/3/library/dataclasses.html)
e [módulos no Python](https://docs.python.org/3/tutorial/modules.html).


[← Caderno anterior](06-calculo-cientifico-com-scipy.ipynb) · [Índice do curso](../README.md) · [Próximo caderno →](08-topicos-complementares.ipynb)

Autor: Diego Tavares Volpatto · [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/deed.pt-br)